<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Supervisor Pattern
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** Der Supervisor übernimmt für den PMO-Assistenten genau die Routing-Entscheidung aus der Leitaufgabe — je nach Fragetyp an einen spezialisierten Worker delegieren, statt einen Generalisten alles machen zu lassen. Die Supervisor-Logik ist **Planen** in Reinform, wird aber erst sinnvoll, wenn die Worker-Rollen fachlich klar getrennt sind. Der `pmo-briefing`-Skill kann in diesem Muster als spezialisierter Worker auftreten, während der Supervisor Recherche, Risiko- und Freigabeschritte koordiniert. Die fachlichen Routen orientieren sich an den drei PMO-Kernfunktionen des Leitprojekts: Meeting-Vorbereitung, Aufgabenverfolgung und Statusreporting.


**Praxis-Transfer:** Mara Vogt profitiert von einem Supervisor, der Tabellenbefunde, Fließtext und Quellenvergleiche an spezialisierte Worker verteilt.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "false"  # nur Referenzlauf per tracing_context (siehe unten)
os.environ["LANGSMITH_PROJECT"] = "m17-supervisor-pattern"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS


# 1 | Übersicht
---

Dieses Notebook vertieft das Supervisor-Pattern für den PMO-Assistenten: Der Supervisor entscheidet nach Fragetyp, ruft spezialisierte Worker auf, begrenzt Schleifen und gibt erst nach Quellen- und Qualitätsprüfung frei.

**Einordnung:** Das Routing ist hier **regelbasiert** (Python-Logik über den State) und nicht LLM-gesteuert. Der Graph leitet anhand seines Zustands an Nodes weiter; ein LLM-Agent, der selbst Worker oder Subagents als Tools auswählt, kommt erst später im Kurs vor (DeepAgents, DeepAgents Harness bis DeepAgents Skill PMO-Assistent).

| Problem im PMO-Assistent | Supervisor-Lösung |
|---|---|
| Tabellenfrage, Fließtextfrage und Vergleichsfrage brauchen unterschiedliche Bearbeitung | Routing nach Fragetyp |
| Worker können unvollständige oder unbelegte Ergebnisse liefern | Kritik- und Quellen-Gate |
| Supervisor-Schleifen können teuer werden | `attempts`, `max_attempts` und `recursion_limit` |
| Debugging ist sonst nur über Modelltext möglich | Routing-Log und strukturierter State |


<p><font color='darkblue' size="4">
ℹ️ <b>Information:</b>
</font></p>

Dieses Notebook nutzt deterministische Worker statt Live-Recherche. Das Pattern bleibt identisch, aber die Ausgabe ist reproduzierbar und der Lauf verursacht keine zusätzlichen Modellkosten außerhalb der Setup- und Trace-Infrastruktur.


In [ ]:
#@markdown <p><font size="4" color='green'>Supervisor im PMO-Assistent</font></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    START([START]) --> SUP["Supervisor
Fragetyp + State prüfen"]
    SUP -->|tabellarisch| TAB["Tabellen-Worker"]
    SUP -->|argumentativ| TXT["Fließtext-Worker"]
    SUP -->|vergleich| CMP["Vergleichs-Worker"]
    SUP -->|Quellen fehlen| SRC["Quellen-Gate"]
    SUP -->|Antwort bereit| SYN["Synthese"]
    SYN --> CRT["Kritik-Gate"]
    CRT --> SUP
    TAB --> SUP
    TXT --> SUP
    CMP --> SUP
    SRC --> SUP
    SUP -->|FINISH| END([END])

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class START input;
    class SUP,SRC,CRT metric;
    class SYN,END output;
'''

mermaid(diagram, width=900)


# 2 | State, Tools und Worker
---

Der robuste Supervisor braucht einen State, der Fortschritt sichtbar macht. Jeder Worker schreibt genau ein fachliches Feld. Der Supervisor routet anhand leerer Felder, Fragetyp und Qualitätsstatus.


In [ ]:
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from IPython.display import Image as IPImage

class SupervisorState(TypedDict):
    query: str
    question_type: str
    route: str
    table_finding: str
    text_finding: str
    comparison_finding: str
    source_status: str
    synthesis: str
    critique: str
    approved: bool
    attempts: int
    max_attempts: int
    log: list[str]


@tool
def extract_table_evidence_demo(query: str) -> str:
    """Liefert einen simulierten Tabellenbefund aus verfügbaren Projektdokumenten."""
    return "Tabellenbefund: Von 5 Aufgaben im Follow-up-Tracker (Stand 19.05.2026) sind 4 erledigt und 1 offen."


@tool
def extract_narrative_evidence_demo(query: str) -> str:
    """Liefert einen simulierten Fließtext_finding aus verfügbaren Projektdokumenten."""
    return "Fließtext_finding: Quellenhinweise und Out-of-Corpus-Regel erhöhen Nachvollziehbarkeit."


@tool
def compare_source_evidence_demo(query: str) -> str:
    """Liefert einen simulierten Vergleichsbefund aus zwei Quellen."""
    return "Vergleichsbefund: Quelle A (Steuerkreis 17.03.2026) nennt D1, Quelle B (Steuerkreis 14.04.2026) nennt D2."


# Quellenregister: Jeder simulierte Befund gehört zu genau einer (oder zwei) Quelle(n) im Korpus
QUELLEN_JE_BEFUND = {
    "table_finding": ["follow_up_tracker.pdf"],
    "text_finding": ["entscheidungsprotokoll_gesamt.pdf"],
    "comparison_finding": ["protokoll_steuerkreis_2026-03-17.pdf", "protokoll_steuerkreis_2026-04-14.pdf"],
}


def quellen_zu_befund(state: SupervisorState) -> list[str]:
    for feld, quellen in QUELLEN_JE_BEFUND.items():
        if state.get(feld):
            return quellen
    return []


def append_log(state: SupervisorState, eintrag: str) -> list[str]:
    return state.get("log", []) + [eintrag]

print("Briefing-State und deterministische Worker-Tools sind bereit.")


<p><font color='darkblue' size="4">
  <b>Wichtig:</b>
</font></p>

Worker sind nur dann sinnvoll, wenn die Zuständigkeit im State sichtbar wird. Ein Tabellen-Worker, der Fließtext schreibt, oder ein Synthese-Worker ohne Quellen-Gate wäre im Kurs kein belastbares Supervisor-Beispiel.

**Grenze der Simulation:** Das Quellenregister `QUELLEN_JE_BEFUND` ordnet jedem simulierten Befund seine Quelle zu, und das Kritik-Gate prüft, dass genau diese Quelle genannt wird. Ob die Quelle die Aussage inhaltlich stützt, prüft es nicht — das leistet erst ein LLM-Kritiker (Reflection, Parallel, Hierarchy, Abschnitt 2) oder die Abfrage im Korpus (RAG).


# 3 | Supervisor mit Iterationsschutz
---

Der Supervisor prüft in fester Reihenfolge: Fragetyp bestimmen, benötigte Worker starten, Quellenstatus prüfen, Synthese formulieren, Kritik-Gate ausführen, dann `FINISH`.

**Hinweis:** Das Kritik-Gate ist die Grundform von Reflection: Es prüft und schickt bei Ablehnung zurück zur Synthese. Anders als bei echter Reflection (Multi-Agent Patterns, Reflection, Parallel, Hierarchy) erhält die Synthese aber keine Kritik als Eingabe und bleibt unverändert; die Schleife endet durch Freigabe oder `max_attempts`. Eine echte Überarbeitung mit Kritikübergabe zeigt Reflection, Parallel, Hierarchy, Abschnitt 2.

In [ ]:
#@markdown <p><font size="4" color='green'>Supervisor mit Guardrails</font></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    START([START]) --> SUP{"Supervisor<br/>attempts < max_attempts?"}

    SUP -->|Nein| END([END])
    SUP -->|Fragetyp fehlt| TYPE["Fragetyp bestimmen"]
    SUP -->|Befund fehlt| WORK["passender Worker"]
    SUP -->|Quelle fehlt| SRC["Quellen-Gate"]
    SUP -->|Synthese fehlt| SYN["Synthese"]
    SUP -->|Kritik fehlt| CRT["Kritik"]
    SUP -->|approved| END

    TYPE --> SUP
    WORK --> SUP
    SRC --> SUP
    SYN --> SUP
    CRT --> SUP

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class START input;
    class SUP,SRC,CRT metric;
    class SYN,END output;
'''

mermaid(diagram, width=760)


In [ ]:
def fragentyp_bestimmen(query: str) -> str:
    frage_klein = query.lower()
    if any(wort in frage_klein for wort in ["tabelle", "kennzahl", "status", "quote"]):
        return "tabellarisch"
    if any(wort in frage_klein for wort in ["vergleiche", "unterschied", "gegenüber"]):
        return "vergleich"
    return "argumentativ"


def supervisor_node(state: SupervisorState) -> dict:
    if state["attempts"] >= state["max_attempts"]:
        return {"route": "FINISH", "log": append_log(state, "Supervisor -> FINISH wegen max_attempts")}

    if not state.get("question_type"):
        route = "fragentyp_node"
    elif state["question_type"] == "tabellarisch" and not state.get("table_finding"):
        route = "tabellen_worker_node"
    elif state["question_type"] == "argumentativ" and not state.get("text_finding"):
        route = "text_worker_node"
    elif state["question_type"] == "vergleich" and not state.get("comparison_finding"):
        route = "vergleich_worker_node"
    elif not state.get("source_status"):
        route = "quellen_gate_node"
    elif not state.get("synthesis"):
        route = "synthese_worker_node"
    elif not state.get("critique"):
        route = "kritik_gate_node"
    elif state.get("approved"):
        route = "FINISH"
    else:
        route = "synthese_worker_node"

    return {"route": route, "log": append_log(state, f"Supervisor -> {route}")}


def supervisor_router(state: SupervisorState) -> Literal["fragentyp_node", "tabellen_worker_node", "text_worker_node", "vergleich_worker_node", "quellen_gate_node", "synthese_worker_node", "kritik_gate_node", END]:
    route = state.get("route", "FINISH")
    return END if route == "FINISH" else route


def fragentyp_node(state: SupervisorState) -> dict:
    typ = fragentyp_bestimmen(state["query"])
    return {"question_type": typ, "attempts": state["attempts"] + 1, "log": append_log(state, f"Fragentyp: {typ}")}


def tabellen_worker_node(state: SupervisorState) -> dict:
    return {"table_finding": extract_table_evidence_demo.invoke({"query": state["query"]}), "attempts": state["attempts"] + 1, "log": append_log(state, "Tabellen-Worker fertig")}


def text_worker_node(state: SupervisorState) -> dict:
    return {"text_finding": extract_narrative_evidence_demo.invoke({"query": state["query"]}), "attempts": state["attempts"] + 1, "log": append_log(state, "Fließtext-Worker fertig")}


def vergleich_worker_node(state: SupervisorState) -> dict:
    return {"comparison_finding": compare_source_evidence_demo.invoke({"query": state["query"]}), "attempts": state["attempts"] + 1, "log": append_log(state, "Vergleichs-Worker fertig")}


def quellen_gate_node(state: SupervisorState) -> dict:
    befund_vorhanden = any([state.get("table_finding"), state.get("text_finding"), state.get("comparison_finding")])
    status = "belegt" if befund_vorhanden else "nicht_belegt"
    return {"source_status": status, "attempts": state["attempts"] + 1, "log": append_log(state, f"Quellen-Gate: {status}")}


def synthese_worker_node(state: SupervisorState) -> dict:
    if state.get("source_status") != "belegt":
        synthesis = "Nicht im Korpus."
    else:
        befund = state.get("table_finding") or state.get("text_finding") or state.get("comparison_finding")
        synthesis = f"Antwort: {befund} Quelle: {', '.join(quellen_zu_befund(state))}."
    return {"synthesis": synthesis, "critique": "", "approved": False, "attempts": state["attempts"] + 1, "log": append_log(state, "Synthese-Worker fertig")}


def kritik_gate_node(state: SupervisorState) -> dict:
    erwartet = quellen_zu_befund(state)
    hat_quelle = bool(erwartet) and all(quelle in state.get("synthesis", "") for quelle in erwartet)
    approved = state.get("source_status") == "belegt" and hat_quelle
    critique = "approved" if approved else "Nacharbeit: Quelle fehlt oder passt nicht zum Befund."
    return {"critique": critique, "approved": approved, "attempts": state["attempts"] + 1, "log": append_log(state, f"Kritik-Gate: {critique}")}

print("Supervisor-Node, Router und Worker-Nodes sind bereit.")


In [ ]:
supervisor_builder = StateGraph(SupervisorState)
supervisor_builder.add_node("supervisor", supervisor_node)
supervisor_builder.add_node("fragentyp_node", fragentyp_node)
supervisor_builder.add_node("tabellen_worker_node", tabellen_worker_node)
supervisor_builder.add_node("text_worker_node", text_worker_node)
supervisor_builder.add_node("vergleich_worker_node", vergleich_worker_node)
supervisor_builder.add_node("quellen_gate_node", quellen_gate_node)
supervisor_builder.add_node("synthese_worker_node", synthese_worker_node)
supervisor_builder.add_node("kritik_gate_node", kritik_gate_node)

supervisor_builder.add_edge(START, "supervisor")
supervisor_builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "fragentyp_node": "fragentyp_node",
        "tabellen_worker_node": "tabellen_worker_node",
        "text_worker_node": "text_worker_node",
        "vergleich_worker_node": "vergleich_worker_node",
        "quellen_gate_node": "quellen_gate_node",
        "synthese_worker_node": "synthese_worker_node",
        "kritik_gate_node": "kritik_gate_node",
        END: END,
    },
)
for node_name in ["fragentyp_node", "tabellen_worker_node", "text_worker_node", "vergleich_worker_node", "quellen_gate_node", "synthese_worker_node", "kritik_gate_node"]:
    supervisor_builder.add_edge(node_name, "supervisor")

supervisor_agent = supervisor_builder.compile()
print("Supervisor-Graph kompiliert.")


In [ ]:
display(IPImage(supervisor_agent.get_graph(xray=True).draw_mermaid_png()))


In [ ]:
initial_state = {
    "query": "Welche Kennzahl zeigt den Stand der Aufgaben im Follow-up-Tracker?",
    "question_type": "",
    "route": "",
    "table_finding": "",
    "text_finding": "",
    "comparison_finding": "",
    "source_status": "",
    "synthesis": "",
    "critique": "",
    "approved": False,
    "attempts": 0,
    "max_attempts": 8,
    "log": [],
}

from langsmith import tracing_context

run_cfg = {"run_name": "m17_kap3_supervisor_grundlauf", "tags": ["supervisor", "pmo-briefing", "m17"]}

# Referenzlauf: nur dieser Aufruf wird in LangSmith erfasst
with tracing_context(enabled=True):
    supervisor_result = supervisor_agent.invoke(initial_state, config={"recursion_limit": 30, **run_cfg, "run_name": "m17_kap3_supervisor_result"})

mprint("**Routing-Log:**")
mprint("\n".join(f"- {eintrag}" for eintrag in supervisor_result["log"]))

mprint("**Synthese:**")
print(supervisor_result["synthesis"])
print("Genehmigt:", supervisor_result["approved"])


# 4 | Iterationsschutz und Kostenkontrolle
---

Supervisor-Systeme brauchen Schutz vor Endlosschleifen und unnötigen Kosten. Der Zähler `attempts` zählt **jede Node-Ausführung** (nicht vollständige Durchläufe); `max_attempts` begrenzt sie, `recursion_limit` ist die technische Notbremse des Graphen. Bei `max_attempts=2` endet der Lauf daher nach Fragetyp-Bestimmung und Worker, noch vor Synthese und Kritik-Gate.

Fachliche Lücken wie fehlender Befund oder fehlende Quelle werden als State sichtbar (`source_status`, `critique`), aber nicht automatisch behoben: Bei Ablehnung läuft die Synthese erneut mit unverändertem Befund.

In [ ]:
#@markdown <p><font size="4" color='green'>Iterationsschutz im Supervisor-System</font></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    WORKER["Worker"] --> TRY{"Befund vorhanden?"}
    TRY -->|Ja| OK["source_status: belegt"]
    TRY -->|Nein| ERR["source_status: nicht_belegt"]

    OK --> SUP["Supervisor plant weiter"]
    ERR --> SUP

    SUP -->|max_attempts erreicht| END([END])
    SUP -->|sonst| SYN["Synthese und Kritik-Gate"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef stop fill:#fee2e2,stroke:#dc2626,stroke-width:2px,color:#7f1d1d;

    %% Zuweisung
    class TRY,SUP metric;
    class OK,END output;
    class ERR stop;
'''

mermaid(diagram, width=720)


In [ ]:
def kontrollierter_lauf(query: str, max_attempts: int = 3) -> dict:
    state = {
        "query": query,
        "question_type": "",
        "route": "",
        "table_finding": "",
        "text_finding": "",
        "comparison_finding": "",
        "source_status": "",
        "synthesis": "",
        "critique": "",
        "approved": False,
        "attempts": 0,
        "max_attempts": max_attempts,
        "log": [],
    }
    return supervisor_agent.invoke(
        state,
        config={
            "run_name": "m17_kap4_supervisor_kontrollierter_lauf",
            "tags": ["guardrail", "m17"],
            "recursion_limit": 20,
        },
    )

limit_result = kontrollierter_lauf("Welche Kennzahl ist relevant?", max_attempts=2)
print("Iterationen:", limit_result["attempts"])
print("Route:", limit_result["route"])
print("Synthese:", limit_result["synthesis"] or "noch nicht erstellt")
assert limit_result["attempts"] <= 2, "max_attempts wurde nicht eingehalten."


<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

Für Kurs- und Produktionsnähe sollte zuerst eine deterministische Kontrolllogik stehen. LLM-Routing kann später ergänzt werden, aber `max_attempts`, Quellen-Gate und kritisierbare State-Felder sollten nicht vom Modelltext abhängen.


# 5 | Debugging mit Trace und Log
---

Für den Kurs reicht nicht, dass eine finale Antwort erscheint. Der Pfad zur Antwort muss sichtbar sein: Supervisor-Entscheidungen, Worker-Ausgaben, Quellenstatus und Kritik-Gate.


In [ ]:
#@markdown <p><font size="4" color='green'>LangSmith Trace-Analyse</font></p>

# Zeigt nur den Referenzlauf: Tracing ist im Setup auf "false" gestellt und nur dort per tracing_context aktiv
try:
    show_trace("m17-supervisor-pattern", limit=3, show_steps=True)
except Exception as exc:
    print("Trace-Anzeige übersprungen:", type(exc).__name__, str(exc)[:180])


# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='darkblue' size="4">
✏️  <b>Notiz:</b>
</font></p>

Die Aufgabenstellungen erweitern den Supervisor des PMO-Assistent. Jeder Selfcheck steht direkt unter der zugehörigen Umsetzung und prüft konkrete State-Felder.

**Grundlagen**

Einen kleinen Supervisor bauen, der mindestens zwei Fragetypen unterscheidet und den passenden Worker startet: Kennzahlfrage an Tabellen-Worker, Begründungsfrage an Fließtext-Worker.


In [ ]:
# Grundlagen: vorhandenen Supervisor als Arbeitsgraph verwenden
my_supervisor_agent = supervisor_agent
run_cfg = {"run_name": "m17_kapA_aufgabe_grundlagen", "tags": ["aufgabe", "grundlagen", "m17"]}

# 1. my_supervisor_agent.invoke() mit einer Kennzahl- ODER Begruendungsfrage aufrufen (mein_result)
# 2. Initialer State: alle Felder aus SupervisorState leer/0/False setzen, max_attempts=8

**Aufbau**

Ein Kritik-Gate ergänzen oder verwenden, das eine Antwort nur genehmigt, wenn Quellenstatus und Quellenhinweis vorhanden sind. Die Anzahl der Iterationen bleibt begrenzt.


In [ ]:
# Aufbau: Kritik-Gate prüfen
run_cfg = {"run_name": "m17_kapA_aufgabe_aufbau", "tags": ["aufgabe", "aufbau", "m17"]}

# 1. my_supervisor_agent.invoke() mit einer Kennzahlfrage aufrufen (aufbau_result), max_attempts=8
# 2. Prüfen: critique == "approved", attempts <= 8, "Quelle:" in synthesis

**Vertiefung**

Das Routing für Vergleichsfragen prüfen. Vergleichsfragen sollen nicht beim einfachen Tabellen-Worker landen, sondern beim Vergleichs-Worker.

In [ ]:
# Vertiefung: Vergleichs-Routing prüfen
run_cfg = {"run_name": "m17_kapA_aufgabe_vertiefung", "tags": ["aufgabe", "vertiefung", "m17"]}

# 1. my_supervisor_agent.invoke() mit einer Vergleichsfrage aufrufen (vertiefung_result), max_attempts=8
# 2. Prüfen: question_type == "vergleich", comparison_finding nicht leer, table_finding leer

<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

Supervisor-Pattern erst einsetzen, wenn Routing wirklich vom Zwischenstand abhängt. Für feste Abläufe bleibt eine einfache Pipeline einfacher. Für wiederverwendbare Teams bieten sich hierarchische Pattern an.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Multi-Agent-Systeme](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/multi-agent-systeme.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
